# Pommerman FFA SR Algorithms

Train and evaluate SR-ADIDAS and Deep SRQ on full four-agent Pommerman FFA. Deep SRQ uses the NfgTransformer SRE solver interface with PATH fallback enabled by default.


In [ ]:
from pathlib import Path
import sys


def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "discrete_action_space").exists() and (path / "relevant_papers").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {current}")

ROOT = _find_repo_root()
for path in (ROOT, ROOT / "discrete_action_space", ROOT / "discrete_action_space" / "bimatrix_game"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from discrete_action_space.pommerman_ffa.notebook_utils import (
    display_evaluation_rollout,
    display_training_reward_plots,
    evaluate_policy,
    policy_from_deep_srq,
    policy_from_sr_adidas,
    train_pommerman_deep_srq,
    train_pommerman_sr_adidas,
)

POMMERMAN_DIR = ROOT / "discrete_action_space" / "pommerman_ffa"

N_EPISODES = 100
MAX_STEPS = 200
EVAL_EPISODES = 20
EVAL_VIDEO_FPS = 4
BATCH_SIZE = 32
SEED = 2025
USE_GPU = True
OUTPUT_ROOT = POMMERMAN_DIR / "runs" / "sr_algorithms"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NFG_CHECKPOINT_PATH = ROOT / "discrete_action_space" / "sre_solvers" / "nfg_transformer" / "nfg_sre_checkpoints" / "nfg_sre_lbf3.pt"
NFG_CHECKPOINT_PATH = NFG_CHECKPOINT_PATH if NFG_CHECKPOINT_PATH.exists() else None
NFG_ACCEPT_GAP = None
NFG_FALLBACK_ENABLED = True


## SR-ADIDAS Training

In [ ]:
sr_adidas_stats = train_pommerman_sr_adidas(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    epsilon_robust=0.5,
    batch_size=BATCH_SIZE,
)


## SR-ADIDAS Training Reward Plots

One scatter plot per agent with mean/std, followed by one combined agent reward curve.

In [ ]:
sr_adidas_training_figs = display_training_reward_plots(sr_adidas_stats)


## SR-ADIDAS Evaluation

Run the trained policy for `EVAL_EPISODES`, show the first rollout video if possible, then plot one boxplot with each agent reward distribution.

In [ ]:
sr_adidas_eval = evaluate_policy(
    policy_from_sr_adidas(sr_adidas_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 5000,
    output_dir=OUTPUT_ROOT / "sr_adidas",
    label="sr_adidas",
)
display_evaluation_rollout(sr_adidas_eval, fps=EVAL_VIDEO_FPS)


## Deep SRQ + NfgTransformer SRE Solver Training

In [ ]:
deep_srq_stats = train_pommerman_deep_srq(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 20,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    epsilon_robust_initial=0.5,
    epsilon_schedule="linear",
    nfg_checkpoint_path=NFG_CHECKPOINT_PATH,
    nfg_accept_gap=NFG_ACCEPT_GAP,
    nfg_fallback_enabled=NFG_FALLBACK_ENABLED,
    batch_size=BATCH_SIZE,
)


## Deep SRQ + NfgTransformer Training Reward Plots

One scatter plot per agent with mean/std, followed by one combined agent reward curve.

In [ ]:
deep_srq_training_figs = display_training_reward_plots(deep_srq_stats)


## Deep SRQ + NfgTransformer Evaluation

Run the trained policy for `EVAL_EPISODES`, show the first rollout video if possible, then plot one boxplot with each agent reward distribution.

In [ ]:
deep_srq_eval = evaluate_policy(
    policy_from_deep_srq(deep_srq_stats["agent"]),
    n_episodes=EVAL_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 6000,
    output_dir=OUTPUT_ROOT / "deep_srq_nfg_transformer",
    label="deep_srq_nfg_transformer",
)
display_evaluation_rollout(deep_srq_eval, fps=EVAL_VIDEO_FPS)
